## setup

In [23]:
# Block 1: Setup
import sys
import os
import time
import json
import pickle
import logging
import warnings
import traceback
from datetime import datetime

# Data & Math
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# OpenML & Hyperparameter Optimization
import openml
import optuna

# Scikit-learn Tools & Processing
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Scikit-learn ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Set plotting style
sns.set(style="whitegrid")

# Warnungen bezüglich liblinear ignorieren
warnings.filterwarnings("ignore", message=".*'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'.*")

In [24]:
EXPERIMENT_CONFIG = {
    "date_of_training": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "suite_name": "OpenML-CC18",

    "datasets": {
        # Sortiert nach Schwierigkeit und Größe (einfachere rausgeworfen für bessere Komplexitätskurven)
        "iris": 61,              # 150 Reihen, 4 Features (Sehr einfach, perfekte Baseline)
        # "credit-g": 31,            # 1000 Reihen, 20 Features (Eher einfach, aber mit mehr Noise)
        # "blood-transfusion": 1464, # 748 Reihen, gutes Noise-Level
        # "phoneme": 1489,           # 5.4k Reihen, 5 Features (Mittel, gute Baseline)
        # "bank-marketing": 1461,    # ~45k Reihen, braucht etwas länger, realistische Daten
        # "nomao": 1486,             # ~34k Reihen, 118 Features (Anspruchsvoll, sehr gute Komplexitäts-Sichtbarkeit)
        # "higgs": 23512,          # (Optional) Kann bei Bedarf einkommentiert werden, aber extrem groß!
        # "mnist_784": 554           # ~70k Reihen, 784 Features (Als Hardcore-Test, falls gewünscht)
    },

    # Predefined models classes (no hyperparameters set yet)
    "models_to_evaluate": [
        ("LogisticRegression", LogisticRegression),
        ("DecisionTreeClassifier", DecisionTreeClassifier),
        ("RandomForestClassifier", RandomForestClassifier),
        ("KNeighborsClassifier", KNeighborsClassifier),
        ("MLPClassifier", MLPClassifier),
    ],
}

### load suite

In [25]:
# Block 2: Loading Datasets, Tasks, and Best Hyperparameters from Benchmark Suite
def _parse_run_parameters(run):
    params = {}
    if not hasattr(run, 'parameter_settings'):
        return params
        
    for param in run.parameter_settings:
        if not isinstance(param, dict): continue
        val = param.get('value') or param.get('oml:value')
        
        if val is not None:
            if val == "null": val = None
            elif val == "true": val = True
            elif val == "false": val = False
            elif val == "NaN": val = float('nan')
            else:
                try: val = eval(val)
                except: pass
                
        p_name = param.get('oml:name', '').replace('clf__', '').replace('classifier__', '').replace('model__', '')
        if p_name and p_name not in ['steps', 'memory', 'verbose', 'numerical', 'categorical']:
            # Fix scikit-learn deprecations internally to prevent fitting crashes
            if p_name == 'max_features' and val == 'auto':
                val = 'sqrt'
                
            params[p_name] = val
    return params

def _extract_run_metrics(run):
    """Extract all available metrics from a run evaluation."""
    metrics = {}
    if hasattr(run, 'evaluations') and isinstance(run.evaluations, dict):
        for metric_name, metric_val in run.evaluations.items():
            # In your print statement you showed that metric_val is just a float e.g., 0.983
            if isinstance(metric_val, (int, float)) and not isinstance(metric_val, bool):
                metrics[f"openml_{metric_name}"] = metric_val
    return metrics

def get_best_hyperparameters(task_id, available_models, eval_measure='predictive_accuracy'):
    best_params = {model_name: {"params": {}, "run_id": None, "openml_score": None, "evaluation_type": eval_measure, "all_openml_metrics": {}} for model_name, _ in available_models}
    
    try:
        evaluations = openml.evaluations.list_evaluations(
            function=eval_measure, tasks=[task_id], output_format='dataframe'
        )
        if evaluations.empty:
            print(f"Task {task_id}: No evaluations found for {eval_measure}.")
            return best_params
            
        # Filter for strictly scikit-learn flows to avoid Weka/R hyperparameter incompatibility
        evaluations = evaluations[evaluations['flow_name'].str.contains('sklearn', case=False, na=False)]
        evaluations = evaluations.sort_values(by='value', ascending=False)
        found_models = set()
        
        for _, eval_row in evaluations.iterrows():
            if len(found_models) == len(available_models): break
                
            flow_name, run_id = eval_row['flow_name'], eval_row['run_id']
            acc = round(eval_row.get('value', 'N/A'), 4) if isinstance(eval_row.get('value'), (float, int)) else 'N/A'
            
            for model_name, _ in available_models:
                if model_name not in found_models and model_name.lower() in flow_name.lower():
                    try:
                        run = openml.runs.get_run(run_id)
                        params = _parse_run_parameters(run)
                        metrics = _extract_run_metrics(run)
                        
                        best_params[model_name] = {"params": params, "run_id": run_id, "openml_score": acc, "evaluation_type": eval_measure, "all_openml_metrics": metrics}
                        found_models.add(model_name)
                        print(f"  Task {task_id} | {model_name} (Run {run_id}) | Metric ({eval_measure}): {acc} | Params: {params}")
                    except: pass
    except Exception as e:
        warnings.warn(f"Task {task_id} generic error: {e}")
        
    return best_params

def _extract_task_info(task_id):
    task = openml.tasks.get_task(task_id)
    eval_measure = getattr(task, 'evaluation_measure', 'predictive_accuracy') or 'predictive_accuracy'
    dataset = task.get_dataset()
    qualities = dataset.qualities  # Extract dataset meta-features
    X, y, _, _ = dataset.get_data(target=task.target_name)
    
    # Dynamically fetch OpenML's predefined train/test split indices
    repeats, folds, samples = task.get_split_dimensions()
    cv_splits = []
    try:
        for f in range(folds):
            train_idx, test_idx = task.get_train_test_split_indices(repeat=0, fold=f)
            cv_splits.append((train_idx, test_idx))
    except Exception as e:
        print(f"Warning: Could not fetch CV splits for task {task_id}: {e}")
            
    return X, y, eval_measure, dataset, qualities, cv_splits, folds, repeats

def load_task_by_dataset_id(dataset_id, name, available_models=None):
    try:
        tasks = openml.tasks.list_tasks(data_id=dataset_id, output_format="dataframe")
        tasks = tasks[tasks['task_type'] == 'Supervised Classification']
        
        if not tasks.empty:
            task_id = int(tasks.iloc[0]['tid'])
            X, y, eval_measure, dataset, qualities, cv_splits, folds, repeats = _extract_task_info(task_id)
            
            print(f"Dataset: {name} (ID: {dataset_id}, Task ID: {task_id}) - Folds: {folds}, Repeats: {repeats}")
            return X, y, task_id, list(y.unique()), get_best_hyperparameters(task_id, available_models, eval_measure) if available_models else {}, cv_splits, qualities
            
    except Exception as e:
        traceback.print_exc()
        
    print(f"Dataset {name} (ID: {dataset_id}): No valid supervised classification task or settings found.")
    return None, None, None, None, {}, [], {}

def initialize_datasets(config):
    dataset_dict = {}
    available_models = config.get("models_to_evaluate", [])
    
    for name, dataset_id in config['datasets'].items():
        X, y, task_id, target_name, best_params, cv_splits, qualities = load_task_by_dataset_id(dataset_id, name, available_models)
        if X is not None:
            dataset_dict[name] = {"X": X, "y": y, "task_id": task_id, "target": target_name, "dataset_id": dataset_id, "best_hyperparameters": best_params, "cv_splits": cv_splits, "qualities": qualities}
            
    return dataset_dict

def preprocess_features(X):
    X = X.copy()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    # Behebt das Problem, dass uint8 (z.B. MNIST Pixel) weggeworfen wird:
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    
    # Numerische Pipeline: Fehlende Werte durch Durchschnitt ersetzen, dann skalieren
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    
    # Kategorische Pipeline: Fehlende Werte durch häufigsten Wert ersetzen, dann One-Hot-Encoding
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

# Execute data loading and update config
EXPERIMENT_CONFIG['dataset_dict'] = initialize_datasets(EXPERIMENT_CONFIG)

Dataset: iris (ID: 61, Task ID: 59) - Folds: 10, Repeats: 1
  Task 59 | MLPClassifier (Run 10590851) | Metric (predictive_accuracy): 0.9733 | Params: {'copy': True, 'with_mean': True, 'with_std': True, 'add_indicator': False, 'fill_value': None, 'missing_values': nan, 'strategy': 'mean', 'activation': 'relu', 'alpha': 0.0001, 'batch_size': 'auto', 'beta_1': 0.9, 'beta_2': 0.999, 'early_stopping': False, 'epsilon': 1e-08, 'hidden_layer_sizes': [100], 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_fun': 15000, 'max_iter': 5000, 'momentum': 0.9, 'n_iter_no_change': 10, 'nesterovs_momentum': True, 'power_t': 0.5, 'random_state': 33404, 'shuffle': True, 'solver': 'adam', 'tol': 0.0001, 'validation_fraction': 0.1, 'warm_start': False}
  Task 59 | LogisticRegression (Run 10228773) | Metric (predictive_accuracy): 0.9733 | Params: {'copy': True, 'fill_value': None, 'missing_values': nan, 'strategy': 'mean', 'feature_range': [0, 1], 'C': 100, 'class_weight': 'balanced', 'dual': F

### complexity

In [26]:
# ---- Model Independent Complexity Measures ----
def rademacher_complexity(model, X, n_iter=100):
    """Approximate Rademacher complexity using model outputs and random +/-1 labels. (Vectorized)"""
    try:
        # Use model output
        if hasattr(model, "decision_function"):
            f = model.decision_function(X)
        elif hasattr(model, "predict_proba"):
            f = model.predict_proba(X)
        else:
            f = model.predict(X)
            
        f = np.array(f)
        if not np.issubdtype(f.dtype, np.number):
            return 0.0

        # Vectorized implementation for huge performance boost instead of looping
        sigmas = np.random.choice([-1, 1], size=(n_iter, *f.shape))
        scores = np.mean(sigmas * f, axis=tuple(range(1, sigmas.ndim)))

        return float(np.mean(np.abs(scores)))
    except Exception:
        return 0.0

def vc_dimension_estimate(model):
    """Rough VC dimension estimate."""
    try:
        if hasattr(model, 'tree_'):
            return int(model.tree_.node_count)
        elif hasattr(model, 'estimators_'):  # RandomForest
            # Use total nodes across all trees as VC proxy
            return sum([est.tree_.node_count for est in model.estimators_])
        elif hasattr(model, 'coef_'):
            return int(model.coef_.shape[1])
        elif hasattr(model, 'n_neighbors'):
            return int(model.n_neighbors)
        elif hasattr(model, 'coefs_') and hasattr(model, 'intercepts_'):
            # MLP: total parameter count
            total_params = sum([coef.size for coef in model.coefs_]) + sum([intercept.size for intercept in model.intercepts_])
            return int(total_params)
        else:
            return 1
    except Exception:
        return 1

def boundary_complexity(model, X, margin_threshold=0.1):
    """Fraction of points close to decision boundary (approximate)."""
    try:
        if hasattr(model, "predict_proba"):
            p = np.array(model.predict_proba(X))
            if p.ndim > 1 and p.shape[1] >= 2:
                # Top 1 - Top 2 probability difference (works seamlessly for binary and multiclass)
                sorted_p = np.sort(p, axis=1)
                margin = sorted_p[:, -1] - sorted_p[:, -2]
            else:
                margin = np.abs(p[:, 0] - 0.5)
            val = float(np.mean(margin < margin_threshold))
            
        elif hasattr(model, "decision_function"):
            m = np.array(model.decision_function(X))
            if m.ndim > 1 and m.shape[1] > 1:
                sorted_m = np.sort(m, axis=1)
                margin = sorted_m[:, -1] - sorted_m[:, -2]
            else:
                margin = np.abs(m)
            val = float(np.mean(margin < margin_threshold))
        else:
            return None
            
        return max(0.0, val)
    except Exception:
        return None

def prediction_entropy(model, X):
    """Entropy of predicted probabilities, higher = more uncertain."""
    try:
        if hasattr(model, "predict_proba"):
            p = model.predict_proba(X)
            val = float(-np.mean(np.sum(p * np.log(p + 1e-12), axis=1)))
            return max(0.0, val)
        return None
    except Exception:
        return None

# ---- Classical (Model Specific) Complexity Measures ----
def evaluate_classical_complexity(model, base_model_name, X_train_processed=None):
    classical_complexity = {}
    
    if base_model_name == "LogisticRegression":
        num_params = float(model.coef_.size + model.intercept_.size) if hasattr(model, 'coef_') and hasattr(model, 'intercept_') else 0
        weight_norm = float(np.linalg.norm(model.coef_)) if hasattr(model, 'coef_') else 0.0
        classical_complexity = {"parameter_count": num_params, "weight_norm": weight_norm}
        
    elif base_model_name == "DecisionTreeClassifier":
        if hasattr(model, 'tree_'):
            classical_complexity = {
                "tree_depth": int(model.get_depth()),
                "leaf_count": int(model.get_n_leaves()),
                "node_count": int(model.tree_.node_count)
            }
        else:
            classical_complexity = {"tree_depth": 0, "leaf_count": 0, "node_count": 0}
            
    elif base_model_name == "RandomForestClassifier":
        if hasattr(model, 'estimators_'):
            depths = [est.get_depth() for est in model.estimators_]
            avg_depth = float(np.mean(depths)) if depths else 0
            total_nodes = sum([est.tree_.node_count for est in model.estimators_])
            classical_complexity = {
                "tree_count": int(len(model.estimators_)),
                "average_tree_depth": avg_depth,
                "total_node_count": int(total_nodes)
            }
        else:
            classical_complexity = {"tree_count": 0, "average_tree_depth": 0.0, "total_node_count": 0}
            
    elif base_model_name == "KNeighborsClassifier":
        classical_complexity = {
            "k_neighbors": int(model.n_neighbors) if hasattr(model, 'n_neighbors') else 0,
            "training_samples_count": int(X_train_processed.shape[0]) if X_train_processed is not None else 0
        }
        
    elif base_model_name == "MLPClassifier":
        if hasattr(model, 'coefs_') and hasattr(model, 'intercepts_'):
            n_layers = int(model.n_layers_)
            # only hidden + output layers
            h_sizes = getattr(model, 'hidden_layer_sizes', [])
            neurons_per_layer = list(h_sizes) if isinstance(h_sizes, (list, tuple)) else [int(h_sizes)]
            neurons_per_layer += [int(model.n_outputs_)]
            total_params = sum([coef.size for coef in model.coefs_]) + sum([intercept.size for intercept in model.intercepts_])
        else:
            n_layers = 0
            neurons_per_layer = []
            total_params = 0
            
        classical_complexity = {
            "layer_count": n_layers,
            "neurons_per_layer": neurons_per_layer,
            "total_parameter_count": total_params
        }
        
    return classical_complexity

### model setup, hyperparameter

In [27]:
import signal

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException("Training exceeded time limit")

if 'run_logs' not in globals():
    run_logs = []

def log_print(message):
    """Gibt Logs in der Konsole aus und speichert sie gleichzeitig für spätere TXT-Dateien."""
    if message == "":
        print("")
        run_logs.append("")
        return
        
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    formatted_message = f"[{timestamp}] {message}"
    print(formatted_message)
    run_logs.append(formatted_message)

# Function to evaluate predefined models
def evaluate_single_model(model_name, ModelClass, X_train, y_train, X_test, y_test, datasets_info, preprocessor, custom_params=None, variant_id="best"):
    """Trains a single model once on the provided train set and evaluates on the test set."""
    
    dataset_qualities = datasets_info.get("qualities", {})
    best_params_dict = datasets_info.get("best_hyperparameters", {})
    
    # Initialize model
    model = ModelClass()
    base_model_name = type(model).__name__
    
    # Apply hyperparameters if available
    hyperparam_info = best_params_dict.get(model_name, {})
    if custom_params is not None:
        best_params = custom_params
    else:
        best_params = hyperparam_info.get("params", {}) if isinstance(hyperparam_info, dict) and "params" in hyperparam_info else hyperparam_info
        
    openml_run_id = hyperparam_info.get("run_id") if isinstance(hyperparam_info, dict) else None
    openml_score = hyperparam_info.get("openml_score") if isinstance(hyperparam_info, dict) else None
    evaluation_type = hyperparam_info.get("evaluation_type", "predictive_accuracy") if isinstance(hyperparam_info, dict) else "predictive_accuracy"
    all_openml_metrics = hyperparam_info.get("all_openml_metrics", {}) if isinstance(hyperparam_info, dict) else {}
    
    if best_params:
        try:
            valid_params = model.get_params().keys()
            filtered_args = {k: v for k, v in best_params.items() if k in valid_params}
            
            # --- PERFORMANCE FIX: Multi-Core wieder aktivieren ---
            if 'n_jobs' in valid_params:
                filtered_args['n_jobs'] = -1
                
            model.set_params(**filtered_args)
        except Exception as e:
            log_print(f"  │  ├─ WARNING: Failed to set params: {e}")
            pass 
            
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', model)
    ])
    
    start_time = time.time()
    
    # Set a 20-minute timeout for pipeline fit and predict
    timeout_seconds = 20 * 60
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout_seconds)
    
    try:
        pipeline.fit(X_train, y_train)
        training_duration = time.time() - start_time
        log_print(f"  │  ├─ SUCCESS TRAINING | Time: {training_duration:.2f}s")
        
        # Metriken auf den Testdaten
        y_pred = pipeline.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        
        # Compute ROC-AUC macro safely
        try:
            n_classes = len(np.unique(y_test))
            if hasattr(pipeline, "predict_proba"):
                y_prob = pipeline.predict_proba(X_test)
                if n_classes == 2 and y_prob.shape[1] == 2:
                    roc = roc_auc_score(y_test, y_prob[:, 1])
                else:
                    roc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
            elif hasattr(pipeline, "decision_function"):
                y_prob = pipeline.decision_function(X_test)
                if n_classes == 2:
                    roc = roc_auc_score(y_test, y_prob)
                else:
                    roc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
            else:
                roc = -1.0
        except Exception:
            roc = -1.0
            
        # --- Complexity Calculation ---
        comp_start = time.time()
        
        # --- FIX: Limit complexity evaluation to max 2000 samples explicitly to prevent metric lockups ---
        sample_size = 2000
        if X_train.shape[0] > sample_size:
            X_train_comp = X_train.sample(n=sample_size, random_state=42)
        else:
            X_train_comp = X_train
            
        X_train_processed = pipeline.named_steps['preprocessor'].transform(X_train_comp)
        fitted_model = pipeline.named_steps['clf']
        
        rad = rademacher_complexity(fitted_model, X_train_processed)
        vc = vc_dimension_estimate(fitted_model)
        bc = boundary_complexity(fitted_model, X_train_processed)
        ent = prediction_entropy(fitted_model, X_train_processed)
        classical = evaluate_classical_complexity(fitted_model, base_model_name, X_train_processed)
        comp_duration = time.time() - comp_start
        
        log_print(f"  │  ├─ SUCCESS COMPLEXITY | Time: {comp_duration:.2f}s")
        log_print(f"  │  └─ SUCCESS TOTAL | Acc: {acc:.4f} | ROC: {roc:.4f}")
        
    except TimeoutException:
        log_print(f"  │  ├─ ERROR: Training exceeded {timeout_seconds/60:.0f} minutes limit!")
        raise TimeoutException("Timeout")
    except Exception as e:
        log_print(f"  │  ├─ ERROR during training: {e}")
        raise e
    finally:
        signal.alarm(0) # Disable alarm
    
    return {
        "model_name": model_name,
        "variant_id": variant_id,
        "base_model": base_model_name,
        "accuracy": acc,
        "roc_macro": roc,
        "precision_macro": prec,
        "recall_macro": rec,
        "f1_macro": f1,
        "n_instances": X_train.shape[0] + X_test.shape[0],
        "n_features": X_train.shape[1],
        "dataset_qualities": dataset_qualities,
        "params": model.get_params(),
        "openml_run_id": openml_run_id,
        "openml_score": openml_score,
        "all_openml_metrics": all_openml_metrics,
        "evaluation_type": evaluation_type,
        "rademacher": rad,
        "vc": vc,
        "boundary": bc,
        "prediction_entropy": ent,
        "classical_complexity": classical,
        "training_duration": training_duration,
        "complexity_duration": comp_duration
    }


### create variations

In [28]:
def get_model_variants(model_name, base_params, task_id, variant_mode="all"):
    """
    Erstellt Varianten um die OpenML-Baseline inkl. neuer Zwischenstufen (A-Varianten).
    'medium' = Baseline.
    Parameter orientieren sich an Tunability (Probst et al. 2019) und Komplexitätsvariation.
    """
    params = base_params.copy() if isinstance(base_params, dict) else {}
    variants = []

    if model_name == "LogisticRegression":
        base_c = float(params.get('C', 1.0))

        mod_params = [
            ("low_2", {"C": base_c * 1e-5, "penalty": "l2"}),
            ("low_2A", {"C": base_c * 1e-4, "penalty": "l2"}),
            ("low_1", {"C": base_c * 1e-2, "penalty": "l2"}),
            ("low_1A", {"C": base_c * 1e-1, "penalty": "l2"}), # mid_low equivalent
            ("mid_lowA", {"C": base_c * 0.5, "penalty": "l2"}),
            ("medium", {"C": base_c, "penalty": "l2"}),
            ("mid_highA", {"C": base_c * 5.0, "penalty": "l2"}),
            ("high_1", {"C": base_c * 1e2, "penalty": "l2"}),
            ("high_1A", {"C": base_c * 1e3, "penalty": "l2"}), # adjusted to fit between high_1 and high_2
            ("high_2A", {"C": base_c * 1e4, "penalty": "l2"}),
            ("high_2", {"C": base_c * 1e5, "penalty": "l2"})
        ]

        for v_id, val_dict in mod_params:
            p = params.copy()
            p.update(val_dict)
            p['max_iter'] = 500
            variants.append((v_id, p))

    elif model_name == "DecisionTreeClassifier":
        base_depth = params.get('max_depth', 10)
        base_min_samples_leaf = params.get('min_samples_leaf', 5)
        base_min_samples_split = params.get('min_samples_split', 2)
        
        if base_depth is None:
            base_depth = 15
        base_depth = int(base_depth)

        mod_params = [
            ("low_2", {"max_depth": max(1, base_depth // 4), "min_samples_leaf": base_min_samples_leaf * 4}),
            ("low_2A", {"max_depth": max(1, base_depth // 5), "min_samples_leaf": base_min_samples_leaf * 5}),
            ("low_1", {"max_depth": max(1, base_depth // 2), "min_samples_leaf": base_min_samples_leaf * 2}),
            ("low_1A", {"max_depth": max(1, base_depth // 3), "min_samples_leaf": base_min_samples_leaf * 3}),
            ("mid_lowA", {"max_depth": int(base_depth * 0.8), "min_samples_leaf": int(base_min_samples_leaf * 1.2)}),
            ("medium", {"max_depth": base_depth, "min_samples_leaf": base_min_samples_leaf, "min_samples_split": base_min_samples_split}),
            ("mid_highA", {"max_depth": int(base_depth * 1.3), "min_samples_leaf": max(1, int(base_min_samples_leaf * 0.8))}),
            ("high_1", {"max_depth": base_depth * 2, "min_samples_leaf": max(1, base_min_samples_leaf // 2)}),
            ("high_1A", {"max_depth": int(base_depth * 2.5), "min_samples_leaf": max(1, base_min_samples_leaf // 3)}),
            ("high_2A", {"max_depth": None, "min_samples_leaf": 2, "min_samples_split": 5}),
            ("high_2", {"max_depth": None, "min_samples_leaf": 1, "min_samples_split": 2})
        ]

        for v_id, val_dict in mod_params:
            p = params.copy()
            p.update(val_dict)
            variants.append((v_id, p))

    elif model_name == "RandomForestClassifier":
        base_estimators = int(params.get('n_estimators', 100))
        base_depth = params.get('max_depth', None)
        base_max_features = params.get('max_features', 'sqrt')

        mod_params = [
            ("low_2", {"max_depth": 2, "max_features": 0.05}),
            ("low_2A", {"max_depth": 3, "max_features": 0.15}),
            ("low_1", {"max_depth": 5, "max_features": 0.2}),
            ("low_1A", {"max_depth": 6, "max_features": 0.3}),
            ("mid_lowA", {"max_depth": 8, "max_features": 0.5}),
            ("medium", {"max_depth": base_depth, "max_features": base_max_features}),
            ("mid_highA", {"max_depth": None, "max_features": 0.7}),
            ("high_1", {"max_depth": None, "max_features": 0.8}),
            ("high_1A", {"max_depth": None, "max_features": 0.9}),
            ("high_2A", {"max_depth": None, "max_features": 1.0}), # practically same as high_2, kept for mapping
            ("high_2", {"max_depth": None, "max_features": 1.0})
        ]

        for v_id, val_dict in mod_params:
            p = params.copy()
            p.update(val_dict)
            p['n_estimators'] = base_estimators  
            variants.append((v_id, p))

    elif model_name == "KNeighborsClassifier":
        base_knn = int(params.get('n_neighbors', 5))
        
        mod_params = [
            ("low_2", {"n_neighbors": base_knn * 10}),
            ("low_2A", {"n_neighbors": min(base_knn * 8, 100)}),
            ("low_1", {"n_neighbors": base_knn * 3}),
            ("low_1A", {"n_neighbors": base_knn * 4}),
            ("mid_lowA", {"n_neighbors": int(base_knn * 1.5)}),
            ("medium", {"n_neighbors": base_knn}),
            ("mid_highA", {"n_neighbors": max(1, int(base_knn / 1.5))}),
            ("high_1", {"n_neighbors": max(1, base_knn // 2)}),
            ("high_1A", {"n_neighbors": max(1, base_knn // 4)}),
            ("high_2A", {"n_neighbors": 2}),
            ("high_2", {"n_neighbors": 1})
        ]

        for v_id, val_dict in mod_params:
            p = params.copy()
            p.update(val_dict)
            variants.append((v_id, p))

    elif model_name == "MLPClassifier":
        base_h = params.get('hidden_layer_sizes', (100,))
        base_alpha = params.get('alpha', 0.0001)

        if isinstance(base_h, int):
            base_h = (base_h,)
        elif isinstance(base_h, list):
            base_h = tuple(base_h)
            
        base_size = base_h[0] if len(base_h) > 0 else 100

        mod_params = [
            ("low_2", {"hidden_layer_sizes": (max(1, base_size // 10),), "alpha": 1.0}),
            ("low_2A", {"hidden_layer_sizes": (max(1, base_size // 8),), "alpha": 0.8}),
            ("low_1", {"hidden_layer_sizes": (max(1, base_size // 3),), "alpha": 0.1}),
            ("low_1A", {"hidden_layer_sizes": (max(1, base_size // 4),), "alpha": 0.2}),
            ("mid_lowA", {"hidden_layer_sizes": (int(base_size / 1.5),), "alpha": 0.01}),
            ("medium", {"hidden_layer_sizes": base_h, "alpha": base_alpha}),
            ("mid_highA", {"hidden_layer_sizes": (int(base_size * 1.5),), "alpha": 5e-4}),
            ("high_1", {"hidden_layer_sizes": (base_size * 2, base_size * 2), "alpha": 1e-5}),
            ("high_1A", {"hidden_layer_sizes": (base_size * 3,), "alpha": 5e-5}),
            ("high_2A", {"hidden_layer_sizes": (base_size * 3, base_size * 3), "alpha": 5e-6}),
            ("high_2", {"hidden_layer_sizes": (base_size * 4, base_size * 4), "alpha": 1e-6})
        ]

        for v_id, val_dict in mod_params:
            p = params.copy()
            p.update(val_dict)
            variants.append((v_id, p))
            
    # Filtern der Varianten basierend auf der Einstellung
    if variant_mode == "simple":
        allowed = ["low_2", "low_2A", "low_1", "low_1A", "mid_lowA", "medium"]
        variants = [v for v in variants if v[0] in allowed]
    elif variant_mode == "complex":
        allowed = ["medium", "mid_highA", "high_1", "high_1A", "high_2A", "high_2"]
        variants = [v for v in variants if v[0] in allowed]

    return variants

## Run

#### calculate hyperparameter and train

In [29]:
# Ensure models_results is global so you don't lose it if you cancel execution!
if 'global_all_results' not in globals():
    global_all_results = []

def save_checkpoint():
    """Speichert einen Zwischenstand sicher auf der Festplatte ab."""
    os.makedirs('result/checkpoints', exist_ok=True)
    backup_file = "result/checkpoints/latest_backup.pkl"
    try:
        with open(backup_file, 'wb') as f:
            pickle.dump(global_all_results, f)
        log_print(f"BACKUP: Auto-saved {len(global_all_results)} combos to disk")
    except Exception as e:
        log_print(f"BACKUP ERROR: Could not save checkpoint: {e}")

def load_checkpoint():
    """Hilfsfunktion: Lädt den letzten Festplatten-Zwischenstand, falls der Jupyter Kernel abstürzt."""
    global global_all_results
    backup_file = "result/checkpoints/latest_backup.pkl"
    if os.path.exists(backup_file):
        with open(backup_file, 'rb') as f:
            global_all_results = pickle.load(f)
        print(f"✅ Restored {len(global_all_results)} results from disk backup!")
    else:
        print("❌ No backup file found.")

def evaluate_model_on_dataset(dataset_name, info, preprocessor, model_name, ModelClass):
    """Isoliert das Training eines einzelnen Modells (inkl. aller Varianten) für einen Datensatz."""
    task_id = info.get('task_id')
    
    # Basis-Parameter ermitteln (für OpenML Baseline Accuracy)
    best_params_dict = info.get("best_hyperparameters", {})
    hyperparam_info = best_params_dict.get(model_name, {})
    base_params = hyperparam_info.get("params", {}) if isinstance(hyperparam_info, dict) and "params" in hyperparam_info else hyperparam_info
    openml_score = hyperparam_info.get("openml_score", "N/A") if isinstance(hyperparam_info, dict) else "N/A"
    
    if isinstance(openml_score, float):
        openml_score_str = f"{openml_score:.4f}"
    else:
        openml_score_str = str(openml_score)
        
    # Baseline Accuracies sind an Modelle gebunden, deswegen loggen wir sie bei der Modellauswahl
    log_print(f"MODEL: {model_name} | OpenML Acc (OpenML): {openml_score_str}")
    
    if not info.get("cv_splits"):
        log_print(f"  └─ SKIP: No CV splits available")
        return []
    
    X, y = info["X"], info["y"]
    results = []
    
    # Varianten holen
    variants = get_model_variants(model_name, base_params, task_id)
    
    fold_accuracies = {v_id: [] for v_id, _ in variants}
    
    for fold_idx, (train_idx, test_idx) in enumerate(info["cv_splits"]):
        log_print(f"  ├─ FOLD {fold_idx}:")
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        for variant_id, custom_params in variants:
            log_print(f"  │  ├─ VARIANT '{variant_id}':")
            try:
                # Trainiere das Modell
                res = evaluate_single_model(
                    model_name, ModelClass, X_train, y_train, X_test, y_test,
                    info, preprocessor, custom_params=custom_params, variant_id=variant_id
                )

                # Füge Meta-Informationen hinzu
                res["dataset"] = dataset_name
                res["task_id"] = task_id
                res["target"] = info.get("target")
                res["dataset_id"] = info.get("dataset_id")
                res["fold"] = fold_idx
                
                fold_accuracies[variant_id].append(res["accuracy"])
                results.append(res)
                    
            except Exception as e:
                log_print(f"  │  │  └─ ERROR: Variant '{variant_id}' failed in fold {fold_idx} | {e}")
                
    # Averages
    log_print(f"  └─ AVERAGES ACROSS {len(info['cv_splits'])} FOLDS:")
    for variant_id, accs in fold_accuracies.items():
        if accs:
            avg_acc = np.mean(accs)
            extra_info = f" | OpenML Baseline: {openml_score_str}" if variant_id == "medium" else ""
            log_print(f"     ├─ Variant '{variant_id}': {avg_acc:.4f} (Avg Acc){extra_info}")
            
    return results

def run_benchmarks(resume=False):
    """
    Hauptschleife zur Orchestrierung aller Datensätze und Modelle.
    Nimmt man resume=True, werden bereits absolvierte Modelle übersprungen.
    """
    global global_all_results, run_logs
    
    if not resume:
        global_all_results.clear()
        run_logs.clear()
        log_print(f"=== STARTING NEW BENCHMARK RUN ===")
    else:
        log_print(f"=== RESUMING BENCHMARK RUN ===")
        log_print(f"Already tracked configurations: {len(global_all_results)}")
        
    # Ermittle bereits verarbeitete (Dataset, Model)-Kombinationen bei Resume
    processed_combos = set()
    if resume:
        for entry in global_all_results:
            processed_combos.add((entry["dataset"], entry["model_name"]))
    
    for dataset_name, info in EXPERIMENT_CONFIG['dataset_dict'].items():
        task_id = info.get('task_id')
        log_print("") # Leere Zeile für bessere Lesbarkeit
        log_print(f"DATASET: {dataset_name} | Task ID: {task_id}")
        
        # Preprocessor nur einmal aufbauen
        preprocessor = preprocess_features(info["X"])

        for model_name, ModelClass in EXPERIMENT_CONFIG["models_to_evaluate"]:
            # Check Resume Condition
            if resume and (dataset_name, model_name) in processed_combos:
                log_print(f"  └─ SKIP: {model_name} (Already evaluated)")
                continue
                
            # Starte abgekapselte Modell-Evaluierung
            new_results = evaluate_model_on_dataset(dataset_name, info, preprocessor, model_name, ModelClass)
            
            if new_results:
                global_all_results.extend(new_results)
                save_checkpoint()

    log_print("")
    log_print(f"=== BENCHMARK RUN COMPLETED ===")

In [30]:
# load_checkpoint()

In [31]:
# --- Auführen des Codes ---
# Setze resume=True ein, wenn du manuell mittendrin abgebrochen hast und einfach dort weitermachen willst!
# Wenn z.B. der Kernel komplett abgestürzt ist, führe vorher einmalig 'load_checkpoint()' einzeln aus!
run_benchmarks(resume=False)

[2026-03-12 08:32:48] === STARTING NEW BENCHMARK RUN ===

[2026-03-12 08:32:48] DATASET: iris | Task ID: 59
[2026-03-12 08:32:48] MODEL: LogisticRegression | OpenML Acc (OpenML): 0.9733
[2026-03-12 08:32:48]   ├─ FOLD 0:
[2026-03-12 08:32:48]   │  ├─ VARIANT 'low_2':
[2026-03-12 08:32:49]   │  ├─ SUCCESS TRAINING | Time: 1.16s
[2026-03-12 08:32:49]   │  ├─ SUCCESS COMPLEXITY | Time: 0.00s
[2026-03-12 08:32:49]   │  └─ SUCCESS TOTAL | Acc: 0.8667 | ROC: 0.9400
[2026-03-12 08:32:49]   │  ├─ VARIANT 'low_2A':
[2026-03-12 08:32:50]   │  ├─ SUCCESS TRAINING | Time: 0.71s
[2026-03-12 08:32:50]   │  ├─ SUCCESS COMPLEXITY | Time: 0.00s
[2026-03-12 08:32:50]   │  └─ SUCCESS TOTAL | Acc: 0.9333 | ROC: 0.9667
[2026-03-12 08:32:50]   │  ├─ VARIANT 'low_1':
[2026-03-12 08:32:50]   │  ├─ SUCCESS TRAINING | Time: 0.63s
[2026-03-12 08:32:50]   │  ├─ SUCCESS COMPLEXITY | Time: 0.00s
[2026-03-12 08:32:50]   │  └─ SUCCESS TOTAL | Acc: 0.9333 | ROC: 1.0000
[2026-03-12 08:32:50]   │  ├─ VARIANT 'low_1A':
[

In [32]:
# load_checkpoint()

In [33]:
trained_models_all = global_all_results

In [34]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

# --- Block 7: Save Best Learnings to CSV & TXT ---
# Prepare rows for the CSVs
training_rows = []
complexity_rows = []

# Fetch a robust consistent datetime for the training run
run_timestamp = EXPERIMENT_CONFIG.get("date_of_training", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

for entry in trained_models_all:
    algo_name = entry.get("model_name", entry.get("model", "Unknown"))
    dataset_id = entry.get("dataset_id")
    task_id = entry.get("task_id")
    variant_id = entry.get("variant_id", "best")

    # 1. Training & Performance Rows
    openml_acc = entry.get("openml_score")
    custom_acc = entry.get("accuracy")
    acc_diff = (custom_acc - openml_acc) if (custom_acc is not None and openml_acc is not None) else None
    
    training_rows.append({
        "algo": algo_name,
        "variant_id": variant_id,
        "fold": entry.get("fold", 0),
        "accuracy_difference": acc_diff,
        "dataset_name": entry.get("dataset"),
        "dataset_id": dataset_id,
        "task_id": task_id,
        "hyperparameter": json.dumps(entry.get("params", {}), cls=NumpyEncoder),
        
        "test_accuracy": custom_acc,
        "openml_accuracy_baseline": openml_acc,
        "test_f1_macro": entry.get("f1_macro"),
        "test_roc_macro": entry.get("roc_macro", -1.0),
        "test_precision_macro": entry.get("precision_macro"),
        "test_recall_macro": entry.get("recall_macro"),
        
        "training_duration": entry.get("training_duration"),
        "date_of_training": run_timestamp
    })
    
    # 2. Complexity Rows
    classical_dict = entry.get("classical_complexity", {})
    classical_val = None
    if algo_name == "LogisticRegression":
        classical_val = classical_dict.get("weight_norm")
    elif algo_name == "DecisionTreeClassifier":
        classical_val = classical_dict.get("node_count")
    elif algo_name == "RandomForestClassifier":
        classical_val = classical_dict.get("total_node_count")
    elif algo_name == "KNeighborsClassifier":
        classical_val = classical_dict.get("k_neighbors")
    elif algo_name == "MLPClassifier":
        classical_val = classical_dict.get("total_parameter_count")

    complexity_rows.append({
        "algo": algo_name,
        "variant_id": variant_id,
        "fold": entry.get("fold", 0),
        "dataset_name": entry.get("dataset"),
        "dataset_id": dataset_id,
        "rademacher_complexity": entry.get("rademacher"),
        "vc_dimension_estimate": entry.get("vc"),
        "boundary_complexity": entry.get("boundary"),
        "prediction_entropy": entry.get("prediction_entropy"),
        "classical_complexity_value": classical_val,
        "classical_complexity_raw": json.dumps(classical_dict, cls=NumpyEncoder)
    })

# Create DataFrames
training_df = pd.DataFrame(training_rows)
complexity_df = pd.DataFrame(complexity_rows)

# Merge beide Tabellen
if training_df.empty or complexity_df.empty:
    print("ACHTUNG: Es wurden keine Ergebnisse für den Export gefunden (die Datensätze sind leer).")
    results_df = pd.DataFrame()
else:
    results_df = pd.merge(
        training_df, 
        complexity_df, 
        on=["algo", "variant_id", "dataset_name", "dataset_id", "fold"]
    )

# Folder-Struktur mit Unterordnern erstellen
now = datetime.now()
time_str = now.strftime('%H-%M-%S_%Y-%m-%d')
base_dir = "result"
csv_dir = os.path.join(base_dir, "csv")
txt_dir = os.path.join(base_dir, "txt")

os.makedirs(csv_dir, exist_ok=True)
os.makedirs(txt_dir, exist_ok=True)

# CSV speichern
if not results_df.empty:
    result_filename = os.path.join(csv_dir, f"experiment_results_{time_str}.csv")
    results_df.to_csv(result_filename, index=False)
else:
    result_filename = "N/A (keine Daten)"

# Logs in TXT speichern (Falls in dieser Session erzeugt durch log_print)
txt_log_filename = os.path.join(txt_dir, f"execution_log_{time_str}.txt")
if 'run_logs' in globals():
    with open(txt_log_filename, "w") as f:
        f.write("\n".join(run_logs))

# Summary in TXT speichern
txt_summary_filename = os.path.join(txt_dir, f"experiment_summary_{time_str}.txt")
with open(txt_summary_filename, "w") as f:
    f.write("=== Experiment Summary ===\n")
    f.write(f"Date of Training: {run_timestamp}\n")
    f.write(f"Export Time:      {now.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total Configurations Evaluated: {len(results_df)}\n")
    f.write("-" * 30 + "\n")
    
    if not results_df.empty:
        f.write(f"Models Evaluated:   {', '.join(results_df['algo'].unique())}\n")
        f.write(f"Datasets Used:      {', '.join(results_df['dataset_name'].unique())}\n")
        f.write(f"Variants Evaluated: {', '.join(results_df['variant_id'].unique())}\n")
    else:
        f.write("Keine Daten vorhanden. Möglicherweise wurde das Training abgebrochen oder nicht gestartet.\n")

print(f"✅ Results cleanly saved to folder: '{base_dir}'")
print(f"  - CSV Data: {result_filename}")
print(f"  - Summary:  {txt_summary_filename}")
print(f"  - Full Log: {txt_log_filename}")

✅ Results cleanly saved to folder: 'result'
  - CSV Data: result/csv/experiment_results_08-34-07_2026-03-12.csv
  - Summary:  result/txt/experiment_summary_08-34-07_2026-03-12.txt
  - Full Log: result/txt/execution_log_08-34-07_2026-03-12.txt
